# Box IoU and Non-Maximum Suppression — Solution

**Difficulty**: 🟢 Easy

### Problem Statement

Every object detector — Faster R-CNN, RetinaNet, YOLO, FCOS, the box head of an open-vocabulary detector — ends with the same post-processing step: the head emits hundreds or thousands of overlapping candidate boxes, and **non-maximum suppression (NMS)** keeps one box per object. NMS is built on **Intersection-over-Union (IoU)**, which also decides positive/negative anchors during training, defines the detection metric (mAP@0.5), is the basis of IoU losses (GIoU/DIoU/CIoU), and associates detections across frames in trackers (SORT, ByteTrack).

Implement from scratch the three functions behind `torchvision.ops.box_iou`, `torchvision.ops.nms` and `torchvision.ops.batched_nms`.

Boxes are `(x1, y1, x2, y2)` with $x_1 \le x_2$ and $y_1 \le y_2$. Zero width or height is allowed, and coordinates may be **negative** (raw regressed boxes often are).

$$\mathrm{IoU}(A, B) = \frac{|A \cap B|}{|A \cup B|} = \frac{|A \cap B|}{|A| + |B| - |A \cap B|}$$

---

### Requirements

1. **`box_iou(boxes1, boxes2) -> Tensor`** — `boxes1` is `(N, 4)`, `boxes2` is `(M, 4)`; return the `(N, M)` matrix with `out[i, j] = IoU(boxes1[i], boxes2[j])`.
   - Fully vectorized with broadcasting — no Python loop over boxes.
   - When the union is 0 (two zero-area boxes) define IoU = 0. Never return NaN.
   - Differentiable with respect to both inputs (no `.item()`, `.tolist()` or re-wrapping in `torch.tensor`): the same function is the core of IoU losses.
   - `N = 0` or `M = 0` returns an empty `(0, M)` / `(N, 0)` tensor.
2. **`nms(boxes, scores, iou_threshold) -> LongTensor`** — greedy NMS:
   - Visit boxes in decreasing score order. A box is **kept** unless an already-kept box has IoU **strictly greater** than `iou_threshold` with it (the torchvision convention). Suppressed boxes never suppress anything.
   - Return the kept indices into the original `boxes`, sorted by decreasing score, as a 1-D `int64` tensor. No boxes → `torch.empty(0, dtype=torch.long)`.
   - Do not modify the inputs.
3. **`batched_nms(boxes, scores, labels, iou_threshold) -> LongTensor`** — NMS run independently per class: a box can only be suppressed by a box with the **same label**. Return the union of the per-class results, sorted by decreasing score. Labels are non-negative integers, not necessarily contiguous. Must stay correct for negative coordinates.

You may assume scores are distinct (ties may be broken arbitrarily).

---

### Constraints

- ✅ Pure PyTorch. `box_iou` must broadcast `(N, 1, ·)` against `(1, M, ·)`.
- ✅ `nms` may loop over boxes in score order (NMS is inherently sequential), but should reuse your vectorized `box_iou`.
- ❌ Do not call `torchvision.ops` (it is only used in the optional cross-check cell).
- ❌ No Python loop over box pairs in `box_iou`.

---

<details>
  <summary>💡 Hint</summary>

  - The intersection of two axis-aligned boxes is itself an axis-aligned box: its top-left corner is the element-wise **max** of the two top-left corners, its bottom-right corner the element-wise **min** of the two bottom-right corners. If the boxes do not overlap, that "box" has negative width or height — what should its area be?
  - Insert a new axis into each input so that a single `torch.maximum` / `torch.minimum` produces all `N × M` corners at once.
  - For NMS, compute the IoU matrix once, then walk the boxes from highest to lowest score while keeping a boolean "suppressed" mask.
  - For `batched_nms`, think about what happens to IoU if you translate boxes. Could you move each class somewhere the other classes can never reach?

</details>

---

In [ ]:
import torch

In [ ]:
# A toy detector output: three objects, six jittered candidate boxes around each.
torch.manual_seed(0)
objects = torch.tensor([[10., 10., 50., 60.],
                        [40., 30., 90., 80.],
                        [120., 20., 160., 70.]])
cand = objects.repeat_interleave(6, dim=0) + 3 * torch.randn(18, 4)
cand = torch.cat([torch.minimum(cand[:, :2], cand[:, 2:]),      # keep x1 <= x2, y1 <= y2
                  torch.maximum(cand[:, :2], cand[:, 2:])], dim=1)
cand_scores = torch.rand(18)
cand_labels = torch.tensor([0] * 6 + [1] * 6 + [0] * 6)          # object 1 is another class
print("candidates:", tuple(cand.shape), "scores:", tuple(cand_scores.shape))


def iou_loop(b1, b2):
    # Brute-force reference: one pair at a time, in plain Python.
    out = torch.zeros(len(b1), len(b2))
    for i, a in enumerate(b1.tolist()):
        for j, b in enumerate(b2.tolist()):
            iw = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
            ih = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
            union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - iw * ih
            out[i, j] = iw * ih / union if union > 0 else 0.0
    return out

In [ ]:
def box_iou(boxes1: torch.Tensor, boxes2: torch.Tensor) -> torch.Tensor:
    """
    Pairwise IoU between two sets of (x1, y1, x2, y2) boxes.

    Args:
        boxes1: (N, 4) boxes.
        boxes2: (M, 4) boxes.

    Returns:
        (N, M) tensor, out[i, j] = IoU(boxes1[i], boxes2[j]); 0 where the union is 0.
    """
    area1 = (boxes1[:, 2] - boxes1[:, 0]) * (boxes1[:, 3] - boxes1[:, 1])      # (N,)
    area2 = (boxes2[:, 2] - boxes2[:, 0]) * (boxes2[:, 3] - boxes2[:, 1])      # (M,)

    # (N, 1, 2) against (1, M, 2) -> corners of all N*M intersections at once.
    top_left = torch.maximum(boxes1[:, None, :2], boxes2[None, :, :2])         # (N, M, 2)
    bottom_right = torch.minimum(boxes1[:, None, 2:], boxes2[None, :, 2:])     # (N, M, 2)
    wh = (bottom_right - top_left).clamp(min=0)    # disjoint boxes -> width or height 0
    inter = wh[..., 0] * wh[..., 1]                                            # (N, M)

    union = area1[:, None] + area2[None, :] - inter                            # (N, M)
    # union == 0 only for two zero-area boxes, where inter == 0 as well. Dividing by 1
    # there returns 0 and, unlike torch.where(union > 0, inter / union, 0), keeps the
    # BACKWARD pass free of 0/0 = NaN (the unselected branch of where still gets a
    # gradient computed, and 0 * NaN = NaN).
    safe_union = torch.where(union > 0, union, torch.ones_like(union))
    return inter / safe_union

In [ ]:
a, b = cand[:7], cand[5:12]
got = box_iou(a, b)
assert got.shape == (7, 7), got.shape
assert torch.allclose(got, iou_loop(a, b), atol=1e-6), "box_iou disagrees with the brute-force loop"
assert torch.allclose(box_iou(objects, objects).diagonal(), torch.ones(3)), "IoU(box, box) must be 1"
assert box_iou(objects[:1], torch.tensor([[200., 200., 210., 210.]])).item() == 0.0, "disjoint -> 0"

point = torch.tensor([[5., 5., 5., 5.]])
assert box_iou(point, point).item() == 0.0, "zero-area boxes: union is 0, IoU is defined as 0 (not NaN)"
assert box_iou(torch.zeros(0, 4), objects).shape == (0, 3)

x = cand[:4].clone().requires_grad_()
box_iou(x, objects).sum().backward()
assert x.grad is not None and torch.isfinite(x.grad).all(), "box_iou must be differentiable"
print("box_iou tests passed")

In [ ]:
def nms(boxes: torch.Tensor, scores: torch.Tensor, iou_threshold: float) -> torch.Tensor:
    """
    Greedy non-maximum suppression.

    Args:
        boxes: (N, 4) boxes in (x1, y1, x2, y2) format.
        scores: (N,) confidence scores.
        iou_threshold: suppress a box when its IoU with a kept, higher-scoring box is > this.

    Returns:
        (K,) int64 indices into `boxes` of the kept boxes, sorted by decreasing score.
    """
    if boxes.shape[0] == 0:
        return torch.empty(0, dtype=torch.long, device=boxes.device)

    order = torch.argsort(scores, descending=True)
    iou = box_iou(boxes, boxes)                                    # (N, N), computed once
    suppressed = torch.zeros(boxes.shape[0], dtype=torch.bool, device=boxes.device)

    keep = []
    for i in order.tolist():               # highest score first
        if suppressed[i]:                  # already explained by a better, KEPT box
            continue
        keep.append(i)
        suppressed |= iou[i] > iou_threshold   # strictly greater, as in torchvision
    return torch.tensor(keep, dtype=torch.long, device=boxes.device)

In [ ]:
def nms_reference(boxes, scores, thr):
    # Textbook greedy NMS: keep a box iff IoU <= thr with every box kept so far.
    order = sorted(range(len(scores)), key=lambda i: -float(scores[i]))
    keep = []
    for i in order:
        if all(float(iou_loop(boxes[i:i + 1], boxes[k:k + 1])) <= thr for k in keep):
            keep.append(i)
    return keep

keep = nms(cand, cand_scores, 0.5)
assert keep.dtype == torch.long and keep.dim() == 1
assert keep.tolist() == nms_reference(cand, cand_scores, 0.5), (keep.tolist(), nms_reference(cand, cand_scores, 0.5))
print("kept", keep.tolist(), "out of", len(cand))

# Strictly greater: IoU([0,0,1,1], [0,0,2,1]) is exactly 0.5, so both survive at 0.5.
two = torch.tensor([[0., 0., 1., 1.], [0., 0., 2., 1.]])
assert nms(two, torch.tensor([0.4, 0.9]), 0.5).tolist() == [1, 0]
assert nms(two, torch.tensor([0.4, 0.9]), 0.49).tolist() == [1]

# A suppresses B; B would suppress C but is already gone, so C survives.
chain = torch.tensor([[0., 0., 10., 10.], [3., 0., 13., 10.], [6., 0., 16., 10.]])
assert nms(chain, torch.tensor([0.9, 0.8, 0.7]), 0.5).tolist() == [0, 2]

empty = nms(torch.zeros(0, 4), torch.zeros(0), 0.5)
assert empty.dtype == torch.long and empty.shape == (0,)
print("nms tests passed")

In [ ]:
def batched_nms(boxes: torch.Tensor, scores: torch.Tensor, labels: torch.Tensor,
                iou_threshold: float) -> torch.Tensor:
    """
    Per-class NMS: boxes with different labels never suppress each other.

    Args:
        boxes: (N, 4) boxes in (x1, y1, x2, y2) format (coordinates may be negative).
        scores: (N,) confidence scores.
        labels: (N,) int64 class ids.
        iou_threshold: as in `nms`.

    Returns:
        (K,) int64 indices of the kept boxes, sorted by decreasing score.
    """
    if boxes.shape[0] == 0:
        return torch.empty(0, dtype=torch.long, device=boxes.device)
    # Coordinate trick: IoU is translation-invariant, so shifting every box of class k
    # by k * span leaves within-class IoUs unchanged, while boxes of different classes
    # land in disjoint regions and can never overlap. One nms call then does all
    # classes. span = max - min + 1 (torchvision uses max + 1, which is only enough
    # separation when every coordinate is >= 0).
    span = boxes.max() - boxes.min() + 1
    offsets = labels.to(boxes.dtype) * span                               # (N,)
    return nms(boxes + offsets[:, None], scores, iou_threshold)

In [ ]:
keep_b = batched_nms(cand, cand_scores, cand_labels, 0.5)
expected = []
for c in cand_labels.unique():
    idx = (cand_labels == c).nonzero().flatten()
    expected += idx[nms(cand[idx], cand_scores[idx], 0.5)].tolist()
expected = sorted(expected, key=lambda i: -float(cand_scores[i]))
assert keep_b.tolist() == expected, (keep_b.tolist(), expected)

# Different labels never suppress each other -- also with negative coordinates.
same = torch.tensor([[-50., -50., 50., 50.]] * 2)
assert batched_nms(same, torch.tensor([0.9, 0.8]), torch.tensor([0, 1]), 0.1).tolist() == [0, 1]
assert batched_nms(same, torch.tensor([0.9, 0.8]), torch.tensor([4, 4]), 0.1).tolist() == [0]
print("batched_nms tests passed")

In [ ]:
# Optional: cross-check against torchvision when it is installed (it is on Colab).
try:
    import torchvision.ops as tvops
except ImportError:
    print("torchvision not installed -- skipping the cross-check")
else:
    g = torch.Generator().manual_seed(1)
    xy = torch.randint(0, 100, (300, 2), generator=g).float()     # integer coords: exact IoU
    boxes = torch.cat([xy, xy + torch.randint(5, 35, (300, 2), generator=g)], dim=1)
    scores = torch.rand(300, generator=g)
    labels = torch.randint(0, 3, (300,), generator=g)
    # torchvision documents 0 <= x1 < x2 and 0 <= y1 < y2, and box_iou returns NaN for
    # two zero-area boxes, so compare on boxes that satisfy that.
    assert torch.allclose(box_iou(boxes, boxes), tvops.box_iou(boxes, boxes), atol=1e-6)
    assert torch.equal(nms(boxes, scores, 0.5), tvops.nms(boxes, scores, 0.5))
    assert torch.equal(batched_nms(boxes, scores, labels, 0.5),
                       tvops.batched_nms(boxes, scores, labels, 0.5))
    print("matches torchvision.ops")

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("iou-nms", box_iou=box_iou, nms=nms, batched_nms=batched_nms)

### Why it works

- **Intersection.** The overlap of two axis-aligned boxes is the box between the element-wise max of the top-left corners and the element-wise min of the bottom-right corners. If the boxes are disjoint, that box has a negative side, so the side lengths are clamped at 0 *before* multiplying (two negative sides would otherwise multiply to a positive "area"). Indexing with `[:, None]` and `[None, :]` broadcasts this to all $N \times M$ pairs in a handful of kernels. The union follows from inclusion–exclusion, $|A \cup B| = |A| + |B| - |A \cap B|$.
- **Greedy NMS.** Visiting boxes from the highest score down, each kept box is the best remaining explanation of its object, and everything that overlaps it by more than the threshold is treated as a duplicate of it. A box that has been suppressed has already been explained by a better box, so it must not suppress anything itself. That is what makes the result depend on the score order, and it is why NMS is sequential.
- **`batched_nms`.** IoU is translation-invariant: shifting both boxes by the same vector changes neither the intersection nor the areas. Shifting class $k$ by $k \cdot \text{span}$ with $\text{span} = \max - \min + 1$ puts each class inside its own region, at least 1 unit away from every other class. The within-class IoUs stay the same and the cross-class IoUs become exactly 0, so one `nms` call is the same as running NMS per class.

### Complexity

- `box_iou`: $O(NM)$ time and memory.
- `nms`: $O(N^2)$ for the IoU matrix plus a Python loop of $N$ steps, each doing $O(N)$ work. That is fine for the usual ≤ a few thousand pre-NMS boxes. torchvision's CUDA kernel computes the IoU matrix as 64-bit masks in parallel tiles, and only the final scan is sequential.
- `batched_nms` with the offset trick runs NMS over all $N$ boxes at once ($O(N^2)$, including cross-class pairs that cannot interact). A loop over classes costs $\sum_c n_c^2$, which is cheaper when there are many classes. torchvision switches to that loop on CPU for large $N$.

### Common bugs

- **No clamp** on the intersection width/height, so disjoint boxes get a positive intersection.
- **`>=` instead of `>`** at the threshold. A pair with IoU exactly 0.5 survives NMS at 0.5 in torchvision.
- **Suppressed boxes suppressing.** The tempting one-liner "keep box *j* if no higher-scoring box has IoU > t with it" is *not* greedy NMS. It drops C in the chain A→B→C even though B was already gone. (Fast NMS from YOLACT does this on purpose, as an approximation that trades accuracy for speed.)
- Returning positions in the **sorted order** instead of indices into the original `boxes`, or returning kept indices sorted by index instead of by score.
- `torch.where(union > 0, inter / union, 0)`: the forward pass is right, but the backward pass gives NaN gradients for degenerate pairs, because the unselected branch still back-propagates $0/0$.
- **Offset trick pitfalls**: `labels * (max + 1)` is only safe for non-negative coordinates. Very large offsets (many classes × large images) also lose float32 precision: at a coordinate of $10^6$ the float32 spacing is 0.0625 px, so IoUs near the threshold can flip.
- Mixing the legacy "+1 pixel" convention (`(x2 - x1 + 1) * (y2 - y1 + 1)`, from old VOC/Detectron code) with continuous coordinates.

### Interview follow-ups

1. **What are GIoU and DIoU, and why do they exist?** An IoU loss has zero gradient when the boxes do not overlap. GIoU $= \mathrm{IoU} - |C \setminus (A \cup B)|/|C|$, where $C$ is the smallest enclosing box, lies in $(-1, 1]$ and still pulls disjoint boxes together. DIoU $= \mathrm{IoU} - \rho^2(\text{centres})/c^2$, with $c$ the diagonal of $C$, penalises centre distance directly and converges faster. CIoU adds an aspect-ratio term. DIoU-NMS uses DIoU as the suppression criterion, so two nearby objects with distant centres both survive.
2. **What is Soft-NMS?** Instead of deleting overlapping boxes it decays their scores, $s_j \leftarrow s_j (1 - \mathrm{IoU})$ (linear, applied above $N_t$) or $s_j \leftarrow s_j\, e^{-\mathrm{IoU}^2/\sigma}$ (Gaussian), and re-picks the maximum after each step. Recall in crowded scenes goes up (occluded people are no longer wiped out) at no training cost. It is still sequential and $O(N^2)$.
3. **Why is NMS non-differentiable, and how does DETR avoid it?** NMS makes hard, discrete keep/drop decisions, so no gradient reaches the duplicate-removal behaviour, and the detector is trained one-to-many and then de-duplicated by a heuristic. DETR treats detection as **set prediction**. Hungarian matching makes a one-to-one assignment between queries and ground truth, and every unmatched query is pushed to "no object". The model learns to output each object once, so no NMS is needed. The cost is slower convergence (fixed by Deformable DETR / DINO / RT-DETR). YOLOv10 gets NMS-free inference with a one-to-one assignment head.
4. **How do you make NMS fast on a GPU?** Compute the full IoU matrix in parallel as bitmasks, then do a cheap sequential scan (torchvision). Alternatively use fully parallel approximations: Fast NMS (column-max of the upper-triangular IoU matrix) or Matrix NMS (SOLOv2), which decays scores in parallel. Both give up exact greedy semantics.
5. **When do you use class-agnostic vs per-class NMS?** Per-class NMS keeps overlapping objects of different classes (a person on a horse) but can output two labels for one ambiguous object. Class-agnostic NMS does the opposite. Many pipelines run per-class NMS and then cap the number of detections per image (e.g. 100).
6. **How do you choose the NMS threshold?** A low threshold merges genuinely overlapping objects (crowds), and a high one leaves duplicates that become false positives in mAP. Typical values are 0.5–0.7. For crowded scenes, use Soft-NMS, adaptive NMS, or set-based (DETR-style) detectors.